# Loading the Data

In [4]:
import os
import json

# Relative path from the notebook to the corpus directory
corpus_dir = '/home/jugal/Documents/DataScience/Advanced_RAG/vectordatabases/open_ragbench/pdf/arxiv/corpus'

# Get all JSON files in the directory and sort them alphabetically 
json_files = sorted([f for f in os.listdir(corpus_dir) if f.endswith('.json')])

# Get the first 100 files
first_100_files = json_files[:100]

loaded_data = []

# Load the content of each JSON file|
for file_name in first_100_files:
    file_path = os.path.join(corpus_dir, file_name)
    with open(file_path, 'r', encoding='utf-8') as f:
        try:
            data = json.load(f)
            loaded_data.append(data)
        except json.JSONDecodeError as e:
            print(f"Error reading {file_name}: {e}")

print(f"Successfully loaded {len(loaded_data)} JSON files.")

# If you want to see the contents of the very first loaded file
# print(loaded_data[0])

import json

id_list = set()

for i in loaded_data:
    id_list.add(i["id"])

with open("/home/jugal/Documents/DataScience/Advanced_RAG/vectordatabases/open_ragbench/pdf/arxiv/qrels.json", "r") as qrels:
    qrels = json.load(qrels)

qrels_100 = {}
for key in qrels:
    if qrels[key]['doc_id'] in id_list:
        qrels_100[key] = qrels[key]

print("len(qrels_100) =", len(qrels_100))

with open("/home/jugal/Documents/DataScience/Advanced_RAG/vectordatabases/open_ragbench/pdf/arxiv/queries.json", "r") as queries:
    queries = json.load(queries)

queries_100 = {}

for qrel in qrels_100:
    queries_100[qrel] = queries[qrel]

print("len(queries_100) =", len(queries_100))


Successfully loaded 100 JSON files.
len(qrels_100) = 334
len(queries_100) = 334


In [3]:
queries_100

{'dc064d11-cd18-4866-8a99-f16b0abec9c6': {'query': 'How does the MLMM approach affect the analysis of Root Mean Squared Error (RMSE)?',
  'type': 'abstractive',
  'source': 'text-image'},
 'b7eb7db7-b4c6-4aac-a44d-bdbd9c1dfd4a': {'query': 'What is the growth pattern for the volume of an n-ball in Euclidean space?',
  'type': 'abstractive',
  'source': 'text'},
 'ecb5322e-468e-4ca1-bcec-57e4404e41af': {'query': 'Do existing methods for AQA explore audio information in videos?',
  'type': 'extractive',
  'source': 'text'},
 '0e2fcafb-3f1c-4ab2-8e85-416c4960eeb7': {'query': 'What is the impact of negative transfer on multi-domain models in T-cell response prediction?',
  'type': 'abstractive',
  'source': 'text-image'},
 '9a9a8ebe-aca2-4c9e-8760-ddfb21585448': {'query': 'How does an auction-managed AMM differ from traditional AMMs?',
  'type': 'abstractive',
  'source': 'text'},
 'bac61451-d99a-43b3-9754-b8a593e5d1d7': {'query': 'Does bounded invariance affect how probability shares are a

# Loading models and Libraries

In [5]:
import json
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from tqdm.auto import tqdm
from collections import defaultdict
import numpy as np
from qdrant_client import models
from sentence_transformers import SentenceTransformer
from fastembed import SparseTextEmbedding

In [9]:
client = QdrantClient(host="localhost", port=6333)

# Evaluation
### Quadrant Connection

In [33]:
key = 'dc064d11-cd18-4866-8a99-f16b0abec9c6'


query = queries_100[key]['query']

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1", device="cpu")

In [ ]:
def evaluate_retrieval():
    k_values = [1, 2, 3, 4, 5]

    metrics = defaultdict(list)

    client = QdrantClient(host="localhost", port=6333)
    collection_name = "research_papers_v1"

    processed = 0

    for key in queries_100:

        query = queries_100[key]["query"]

        query_vector = embedding_model.encode(query)

        search_results = client.query_points(
            collection_name=collection_name,
            query=query_vector,
            limit=max(k_values),
            with_payload=True,
            with_vectors=False
        ).points

        retrieved_doc_ids = []

        for point in search_results:

            doc_id = point.payload.get("section_id") if point.payload else None

            if not doc_id:
                doc_id = point.id

            retrieved_doc_ids.append(str(doc_id))

        # Ground Truth
        ground_truth = qrels[key]
        ground_truth_doc_id = (
            str(ground_truth["section_id"]) +
            ground_truth["doc_id"]
        )

        # Recall@K
        for k in k_values:

            retrieved_k = retrieved_doc_ids[:k]

            hit = int(
                ground_truth_doc_id in retrieved_k
            )

            metrics[f"Recall@{k}"].append(hit)

        # Standard MRR
        rr = 0.0

        if ground_truth_doc_id in retrieved_doc_ids:
            rank = (
                retrieved_doc_ids.index(
                    ground_truth_doc_id
                ) + 1
            )
            rr = 1.0 / rank

        metrics["MRR"].append(rr)

        processed += 1

        if processed % 20 == 0:
            print(
                f"Processed {processed}/{len(queries_100)} queries..."
            )

    final_metrics = {
        metric: round(np.mean(values), 4)
        for metric, values in metrics.items()
    }

    return final_metrics

In [80]:
metrics = evaluate_retrieval()

Processed 20/334 queries...
Processed 40/334 queries...
Processed 60/334 queries...
Processed 80/334 queries...
Processed 100/334 queries...
Processed 120/334 queries...
Processed 140/334 queries...
Processed 160/334 queries...
Processed 180/334 queries...
Processed 200/334 queries...
Processed 220/334 queries...
Processed 240/334 queries...
Processed 260/334 queries...
Processed 280/334 queries...
Processed 300/334 queries...
Processed 320/334 queries...


# Using dence Vector Results

- 'Recall@1': np.float64(0.5958),
- 'Recall@2': np.float64(0.7246),
- 'Recall@3': np.float64(0.7904),
-  'Recall@4': np.float64(0.8443),
-  'Recall@5': np.float64(0.8653),
-  'MRR': np.float64(0.6998)}


###  MRR

An MRR of 0.6998 means that, on average, the correct document is being retrieved very close to the top of the ranking.

### Recall 

The sweet spot of recall is at Recall @4​


# Hybrid rag Evaluation

In [18]:
def evaluate_hybrid_retrieval():
    k_values = [1, 2, 3, 4, 5]
    metrics = defaultdict(list)

    # 1. Initialize Clients
    # Load the sparse model (dense 'model' is assumed to be globally available like before)
    bm25_model = SparseTextEmbedding(model_name="Qdrant/bm25")
    collection_name = "research_papers_v2_hybrid"
    image = 0

    embedding_model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1", device="cpu")
    processed = 0
    # from sentence_transformers import SentenceTransformer

    # embedding_model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1", device="cpu")

    for key in queries_100:
        query = queries_100[key]["query"]

        # 2. Generate Vectors
        dense_query_vector = embedding_model.encode(query)
        
        sparse_emb = list(bm25_model.embed([query]))[0]
        sparse_query_vector = models.SparseVector(
            indices=sparse_emb.indices.tolist(),
            values=sparse_emb.values.tolist()
        )

        # 3. Hybrid Search using RRF
        search_results = client.query_points(
            collection_name=collection_name,
            prefetch=[
                # Fetch Top 5 using Dense Embeddings
                models.Prefetch(
                    query=dense_query_vector,
                    using="dense",
                    limit=10,
                ),
                # Fetch Top 5 using Sparse (BM25) Embeddings
                models.Prefetch(
                    query=sparse_query_vector,
                    using="sparse",
                    limit=10,
                ),
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=5,
            with_payload=True,
            with_vectors=False
        ).points

        retrieved_doc_ids = []

        # Extract doc_ids as before
        for point in search_results:
            doc_id = point.payload.get("section_id") if point.payload else None
            if not doc_id:
                doc_id = point.id
            retrieved_doc_ids.append(str(doc_id))

            is_image = point.payload.get("image_path", None) if point.payload else None

            if is_image:
                image += 1
        
        # Ground Truth Evaluation
        ground_truth = qrels[key]
        ground_truth_doc_id = (
            str(ground_truth["section_id"]) + ground_truth["doc_id"]
        )

        # Recall@K
        for k in k_values:
            retrieved_k = retrieved_doc_ids[:k]
            hit = int(ground_truth_doc_id in retrieved_k)
            metrics[f"Recall@{k}"].append(hit)

        # Standard MRR
        rr = 0.0
        if ground_truth_doc_id in retrieved_doc_ids:
            rank = retrieved_doc_ids.index(ground_truth_doc_id) + 1
            rr = 1.0 / rank

        metrics["MRR"].append(rr)

        processed += 1
        if processed % 20 == 0:
            print(f"Processed {processed}/{len(queries_100)} queries...")

    # Calculate final means
    final_metrics = {
        metric: round(np.mean(values), 4)
        for metric, values in metrics.items()
    }

    return image

# Run the evaluation
image_count = evaluate_hybrid_retrieval()
print("Image Counts:", image_count)


Processed 20/334 queries...
Processed 40/334 queries...
Processed 60/334 queries...
Processed 80/334 queries...
Processed 100/334 queries...
Processed 120/334 queries...
Processed 140/334 queries...
Processed 160/334 queries...
Processed 180/334 queries...
Processed 200/334 queries...
Processed 220/334 queries...
Processed 240/334 queries...
Processed 260/334 queries...
Processed 280/334 queries...
Processed 300/334 queries...
Processed 320/334 queries...
Image Counts: 0


## Hybrid search Results(for Top 5)

Latency = 1 minutes 11 secs // 334 Queries

- 'Recall@1': np.float64(0.6766), 
- 'Recall@2': np.float64(0.8084), 
- 'Recall@3': np.float64(0.8772), 
- 'Recall@4': np.float64(0.9192), 
- 'Recall@5': np.float64(0.9371), 
- 'MRR': np.float64(0.7795)

## Hybrid search Results(for Top 10)

Latency = 1 minutes 15 secs // 334 Queries

- 'Recall@1': np.float64(0.7156)
- 'Recall@2': np.float64(0.8263) 
- 'Recall@3': np.float64(0.8922)
- 'Recall@4': np.float64(0.9192) 
- 'Recall@5': np.float64(0.9341) -
- 'MRR': np.float64(0.8085)

**Increasing the more number retrival chunks didn't increse the Results**

### Further Steps

- Cross Encoder
- Weighted fusion
- Relative Score Fusion
- Learned Sparse Embeddings (SPLADE)\
- Alpha Tunning
- Query Adaptive Alpha
- Retrieval Depth
- Reranker Models
- Late Interactiona and Colbert

# Cross Encoders

In [4]:
from FlagEmbedding import FlagReranker

reranker_model = FlagReranker(
    "BAAI/bge-reranker-v2-m3",
    use_fp16=False,
    devices="cpu"
)

In [5]:
print(reranker_model.target_devices)

['cpu']


Calculation:

for 50 chunks
- 334 queries
- 50 chunks retrived
- total pairs = 50 * 334 = 16700
- latency per pair = 3.1 sec
- Total Latency = 14 hours approx.

for 10 chunks
- 334 queries
- 10 chunks retrived
- total pairs = 10 * 334 = 3440
- latency per pair = 3.1 sec
- Total Latency = 2 hours approx.

In [27]:
10 * 20 * 3.1 

620.0

In [9]:
text = """Elon Reeve Musk (/ˈiːlɒn/ ⓘ EE-lon; born June 28, 1971) is a businessman and former public official who is the CEO and largest shareholder of Tesla and SpaceX. Musk has been the wealthiest person in the world since 2025, and became the first and only trillionaire in terms of US dollars in mid June 2026; however as of June 21, 2026, Forbes estimates his net worth to be US$951 billion.

Born into the wealthy Musk family in Pretoria, South Africa, Musk emigrated in 1989 to Canada; he has Canadian citizenship since his mother was born there. He received bachelor's degrees in 1997 from the University of Pennsylvania before moving to California to pursue business ventures. In 1995, Musk co-founded Zip2, a web software company. Following its sale in 1999, he co-founded X.com, an e-commerce payment system that merged with Confinity in March 2000 to form PayPal, which was acquired by eBay in 2002. Musk also became an American citizen in 2002.

In 2002, Musk founded and became CEO and chief engineer of SpaceX, a space technology company; the company has since led innovations in reusable rockets and commercial spaceflight. Musk joined Tesla as an early investor in 2004 and became its CEO and product architect in 2008; it has since become a leader in electric vehicles. In 2015, Musk co-founded OpenAI to advance artificial intelligence (AI) research, but later left; his growing discontent with the organization's direction and leadership in the AI boom in the 2020s led him to establish xAI, which became a subsidiary of SpaceX in 2026. In 2022, he acquired Twitter, a social networking service; he implemented significant changes and rebranded it as X in 2023. His other businesses include Neuralink, a neurotechnology company that he co-founded in 2016, and the Boring Company, a tunneling company that he founded in 2017. In November 2025, Tesla approved a pay package worth $1 trillion for Musk, which he is to receive over 10 years if certain milestones are met, such as achieving a market capitalization of $8.5 trillion.

Musk is a supporter of global far-right politics, figures, and political parties. He was the largest donor in the 2024 U.S. presidential election, where he supported Donald Trump. After Trump was inaugurated in January 2025, Musk served as Senior Advisor to the President and as the de facto head of the Department of Government Efficiency (DOGE). Musk left the Trump administration in May 2025 and returned to managing his companies; shortly thereafter he had a public feud with Trump.

Musk's political activities, statements and views have made him a polarizing figure. He has been criticized for making unscientific and misleading statements, including spreading COVID-19 misinformation, promoting conspiracy theories, and affirming antisemitic, white nationalist, racist, and transphobic comments. His acquisition of Twitter was controversial because, following his pledge to decrease censorship, there was an increase in hate speech and misinformation on the service. His role in the second Trump administration attracted public backlash, particularly in response to DOGE and its cuts to the US Agency for International Development (USAID).
"""
text = text[:1500]
query = "When does elon musk created SpaseX"

In [21]:
reranker_model.compute_score([text,query])

[2.1324703693389893]

In [ ]:
def evaluate_hybrid_retrieval():
    k_values = [1, 2, 3, 4, 5]
    metrics = defaultdict(list)

    # 1. Initialize Clients
    # Load the sparse model (dense 'embedding_model' is assumed to be globally available like before)
    bm25_model = SparseTextEmbedding(model_name="Qdrant/bm25")
    
    # Use the local path instead of localhost server
    client = QdrantClient(host="localhost", port=6333)
    collection_name = "research_papers_v2_hybrid"

    processed = 0

    for key in queries_100:
        query = queries_100[key]["query"]

        # 2. Generate Vectors (Updated to use embedding_model)
        dense_query_vector = embedding_model.encode(query)
        
        sparse_emb = list(bm25_model.embed([query]))[0]
        sparse_query_vector = models.SparseVector(
            indices=sparse_emb.indices.tolist(),
            values=sparse_emb.values.tolist()
        )

        # 3. Hybrid Search using RRF (Take top 50)
        search_results = client.query_points(
            collection_name=collection_name,
            prefetch=[
                # Fetch Top 50 using Dense Embeddings
                models.Prefetch(
                    query=dense_query_vector,
                    using="dense",
                    limit=50,
                ),
                # Fetch Top 50 using Sparse (BM25) Embeddings
                models.Prefetch(
                    query=sparse_query_vector,
                    using="sparse",
                    limit=50,
                ),
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=10,
            with_payload=True,
            with_vectors=False
        ).points
        
        # 4. Rerank the top 50 results
                # 4. Rerank the top 50 results
        text_pairs = []
        for point in search_results:
            text = point.payload.get("text", "") if point.payload else ""
            text_pairs.append([query, text])

        # Compute reranker scores
        scores = reranker_model.compute_score(text_pairs)

        # Sort by reranker score (higher is better)
        reranked_results = sorted(
            zip(search_results, scores),
            key=lambda x: x[1],
            reverse=True
        )

        top_5_results = [result[0] for result in reranked_results[:max(k_values)]]
        retrieved_doc_ids = []

        # Extract doc_ids as before
        for point in top_5_results:
            doc_id = point.payload.get("section_id") if point.payload else None
            if not doc_id:
                doc_id = point.id
            retrieved_doc_ids.append(str(doc_id))

        # Ground Truth Evaluation
        ground_truth = qrels[key]
        ground_truth_doc_id = (
            str(ground_truth["section_id"]) + ground_truth["doc_id"]
        )

        # Recall@K
        for k in k_values:
            retrieved_k = retrieved_doc_ids[:k]
            hit = int(ground_truth_doc_id in retrieved_k)
            metrics[f"Recall@{k}"].append(hit)

        # Standard MRR
        rr = 0.0
        if ground_truth_doc_id in retrieved_doc_ids:
            rank = retrieved_doc_ids.index(ground_truth_doc_id) + 1
            rr = 1.0 / rank

        metrics["MRR"].append(rr)

        processed += 1
        if processed % 20 == 0:
            print(f"Processed {processed}/{len(queries_100)} queries...")

    # Calculate final means
    final_metrics = {
        metric: round(np.mean(values), 4)
        for metric, values in metrics.items()
    }

    return final_metrics

# Run the evaluation
hybrid_metrics = evaluate_hybrid_retrieval()
print("Hybrid Search Results:", hybrid_metrics)


Fetching 18 files: 100%|██████████| 18/18 [00:01<00:00, 12.45it/s]


Processed 20/334 queries...
Processed 40/334 queries...
Processed 60/334 queries...
Processed 80/334 queries...
Processed 100/334 queries...
Processed 120/334 queries...
Processed 140/334 queries...
Processed 160/334 queries...
Processed 180/334 queries...
Processed 200/334 queries...
Processed 220/334 queries...
Processed 240/334 queries...
Processed 260/334 queries...
Processed 280/334 queries...
Processed 300/334 queries...
Processed 320/334 queries...
Hybrid Search Results: {'Recall@1': np.float64(0.7455), 'Recall@2': np.float64(0.8653), 'Recall@3': np.float64(0.9012), 'Recall@4': np.float64(0.9281), 'Recall@5': np.float64(0.9491), 'MRR': np.float64(0.8283)}


In [28]:
hybrid_metrics

{'Recall@1': np.float64(0.7455),
 'Recall@2': np.float64(0.8653),
 'Recall@3': np.float64(0.9012),
 'Recall@4': np.float64(0.9281),
 'Recall@5': np.float64(0.9491),
 'MRR': np.float64(0.8283)}

# Putting the Results Together

## Using dence Vector Results

- 'Recall@1': np.float64(0.5958),
- 'Recall@2': np.float64(0.7246),
- 'Recall@3': np.float64(0.7904),
-  'Recall@4': np.float64(0.8443),
-  'Recall@5': np.float64(0.8653),
-  'MRR': np.float64(0.6998)

## Hybrid search Results

Latency = 3 minutes 21 secs // 334 Queries

- 'Recall@1': np.float64(0.6766), 
- 'Recall@2': np.float64(0.8084), 
- 'Recall@3': np.float64(0.8772), 
- 'Recall@4': np.float64(0.9192), 
- 'Recall@5': np.float64(0.9371), 
- 'MRR': np.float64(0.7795)


## Cross Encoder

Latency = 161 min for 10 chunks retrival per query

- 'Recall@1': np.float64(0.7455),
- 'Recall@2': np.float64(0.8653),
- 'Recall@3': np.float64(0.9012),
- 'Recall@4': np.float64(0.9281),
- 'Recall@5': np.float64(0.9491),
- 'MRR': np.float64(0.8283)

# Colbert lateinteraction

In [5]:
from qdrant_client import QdrantClient
from qdrant_client import models
from fastembed import LateInteractionTextEmbedding, SparseTextEmbedding
from collections import defaultdict
import numpy as np

def evaluate_colbert_hybrid_retrieval():
    k_values = [1, 2, 3, 4, 5]
    metrics = defaultdict(list)

    # 1. Initialize Clients and Models
    print("Initializing models...")
    colbert_model = LateInteractionTextEmbedding(model_name="colbert-ir/colbertv2.0")
    bm25_model = SparseTextEmbedding(model_name="Qdrant/bm25")
    
    client = QdrantClient(host="localhost", port=6333)
    collection_name = "research_papers_v3_Colbert" # Use the newly created collection

    processed = 0

    for key in queries_100:
        query = queries_100[key]["query"]

        # 2. Generate Vectors
        # Important: Use query_embed() for LateInteraction queries
        colbert_emb = list(colbert_model.query_embed(query))[0]
        # Ensure it's a python list of lists for Qdrant
        colbert_query_vector = colbert_emb.tolist() if hasattr(colbert_emb, 'tolist') else colbert_emb
        
        # Sparse (BM25) embedding
        sparse_emb = list(bm25_model.query_embed(query))[0] # query_embed also recommended for sparse
        sparse_query_vector = models.SparseVector(
            indices=sparse_emb.indices.tolist() if hasattr(sparse_emb.indices, 'tolist') else sparse_emb.indices,
            values=sparse_emb.values.tolist() if hasattr(sparse_emb.values, 'tolist') else sparse_emb.values,
        )

        # 3. Hybrid Search using RRF (No Cross-Encoder needed!)
        search_results = client.query_points(
            collection_name=collection_name,
            prefetch=[
                # Fetch Top 10 using ColBERT
                models.Prefetch(
                    query=colbert_query_vector,
                    using="colbert",
                    limit=10,
                ),
                # Fetch Top 10 using Sparse (BM25) Embeddings
                models.Prefetch(
                    query=sparse_query_vector,
                    using="bm25",
                    limit=10,
                ),
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=max(k_values), # Directly limit to Top 5 since ColBERT handles the deep interaction
            with_payload=True,
            with_vectors=False
        ).points

        # 4. Extract doc_ids directly from the RRF results
        retrieved_doc_ids = []
        for point in search_results:
            doc_id = point.payload.get("section_id") if point.payload else None
            if not doc_id:
                doc_id = point.id
            retrieved_doc_ids.append(str(doc_id))

        # Ground Truth Evaluation
        ground_truth = qrels[key]
        ground_truth_doc_id = (
            str(ground_truth["section_id"]) + ground_truth["doc_id"]
        )

        # Recall@K
        for k in k_values:
            retrieved_k = retrieved_doc_ids[:k]
            hit = int(ground_truth_doc_id in retrieved_k)
            metrics[f"Recall@{k}"].append(hit)

        # Standard MRR
        rr = 0.0
        if ground_truth_doc_id in retrieved_doc_ids:
            rank = retrieved_doc_ids.index(ground_truth_doc_id) + 1
            rr = 1.0 / rank

        metrics["MRR"].append(rr)

        processed += 1
        if processed % 20 == 0:
            print(f"Processed {processed}/{len(queries_100)} queries...")

    # Calculate final means
    final_metrics = {
        metric: round(np.mean(values), 4)
        for metric, values in metrics.items()
    }

    return final_metrics

# Run the evaluation
colbert_hybrid_metrics = evaluate_colbert_hybrid_retrieval()
print("ColBERT Hybrid Search Results:", colbert_hybrid_metrics)


Initializing models...
Processed 20/334 queries...
Processed 40/334 queries...
Processed 60/334 queries...
Processed 80/334 queries...
Processed 100/334 queries...
Processed 120/334 queries...
Processed 140/334 queries...
Processed 160/334 queries...
Processed 180/334 queries...
Processed 200/334 queries...
Processed 220/334 queries...
Processed 240/334 queries...
Processed 260/334 queries...
Processed 280/334 queries...
Processed 300/334 queries...
Processed 320/334 queries...
ColBERT Hybrid Search Results: {'Recall@1': np.float64(0.7096), 'Recall@2': np.float64(0.8473), 'Recall@3': np.float64(0.8862), 'Recall@4': np.float64(0.9222), 'Recall@5': np.float64(0.9311), 'MRR': np.float64(0.8022)}


In [7]:
colbert_hybrid_metrics

{'Recall@1': np.float64(0.7096),
 'Recall@2': np.float64(0.8473),
 'Recall@3': np.float64(0.8862),
 'Recall@4': np.float64(0.9222),
 'Recall@5': np.float64(0.9311),
 'MRR': np.float64(0.8022)}

## Using dence Vector Results

- 'Recall@1': np.float64(0.5958),
- 'Recall@2': np.float64(0.7246),
- 'Recall@3': np.float64(0.7904),
-  'Recall@4': np.float64(0.8443),
-  'Recall@5': np.float64(0.8653),
-  'MRR': np.float64(0.6998)

## Hybrid search Results(for Top 10)

Latency = 1 minutes 15 secs // 334 Queries

- 'Recall@1': np.float64(0.7156)
- 'Recall@2': np.float64(0.8263) 
- 'Recall@3': np.float64(0.8922)
- 'Recall@4': np.float64(0.9192) 
- 'Recall@5': np.float64(0.9341) -
- 'MRR': np.float64(0.8085)


## Cross Encoder

Latency = 161 min for 10 chunks retrival per query

- 'Recall@1': np.float64(0.7455),
- 'Recall@2': np.float64(0.8653),
- 'Recall@3': np.float64(0.9012),
- 'Recall@4': np.float64(0.9281),
- 'Recall@5': np.float64(0.9491),
- 'MRR': np.float64(0.8283)


## Colbert
Latency = 1 min overall
- 'Recall@1': np.float64(0.7096),
- 'Recall@2': np.float64(0.8473),
- 'Recall@3': np.float64(0.8862),
- 'Recall@4': np.float64(0.9222),
- 'Recall@5': np.float64(0.9311),
- 'MRR': np.float64(0.8022)

## Observation

- Cross Encoder is only worth It, For Top 1 chunk 
- For 2, 3, 4 and 5 chunks Hybrid Search results are giving enough results.
- Only Dense Retrival should not be used and ColBert is not seems to be very good choice
- If we find a way to getting the Good chunk to go above then the Hybrid search results can be as good as Cross Encoder with low cost and latency


In [14]:
import numpy as np
import pandas as pd
from collections import defaultdict
from qdrant_client import QdrantClient, models

def evaluate_weighted_fusion():
    k_values = [1, 2, 3, 4, 5]
    embedding_model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1", device="cpu")
    
    # 1. Initialize Client
    client = QdrantClient(host="localhost", port=6333)
    bm25_model = SparseTextEmbedding(model_name="Qdrant/bm25")
    collection_name = "research_papers_v2_hybrid"

    # Pre-fetch all query results to avoid redundant Qdrant calls across weights
    query_results = {}
    processed = 0
    
    print("Fetching initial results from Qdrant...")
    for key in queries_100:
        query = queries_100[key]["query"]

        dense_query_vector = embedding_model.encode(query)
        sparse_emb = list(bm25_model.embed([query]))[0]
        sparse_query_vector = models.SparseVector(
            indices=sparse_emb.indices.tolist(),
            values=sparse_emb.values.tolist()
        )

        # Get top 20 using dense embeddings
        dense_res = client.query_points(
            collection_name=collection_name,
            query=dense_query_vector,
            using="dense",
            limit=10,
            with_payload=True
        ).points
        
        # Get top 20 using sparse embeddings
        sparse_res = client.query_points(
            collection_name=collection_name,
            query=sparse_query_vector,
            using="sparse",
            limit=10,
            with_payload=True
        ).points
        
        query_results[key] = {
            "dense": dense_res,
            "sparse": sparse_res
        }
        
        processed += 1
        if processed % 50 == 0:
            print(f"Fetched {processed}/{len(queries_100)} queries...")

    # 2. Iterate over all weights [0.1, 0.2, 0.3, ... 0.9]
    weights = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    all_metrics = []

    print("\nEvaluating weighted fusion...")
    for w in weights:
        metrics = defaultdict(list)
        
        for key in queries_100:
            dense_res = query_results[key]["dense"]
            sparse_res = query_results[key]["sparse"]
            
            # Helper to apply Min-Max Normalization
            def process_results(res):
                scores = [p.score for p in res]
                if not scores:
                    return {}
                min_s, max_s = min(scores), max(scores)
                
                norm_dict = {}
                for p in res:
                    doc_id = str(p.payload.get("section_id", p.id) if p.payload else p.id)
                    # Normalize between 0 and 1
                    norm = (p.score - min_s) / (max_s - min_s) if max_s > min_s else 1.0
                    norm_dict[doc_id] = max(norm_dict.get(doc_id, 0), norm)
                return norm_dict
            
            dense_norm = process_results(dense_res)
            sparse_norm = process_results(sparse_res)
            
            # Union of all retrieved document IDs
            all_docs = set(dense_norm.keys()).union(set(sparse_norm.keys()))
            
            # Calculate final fused score
            final_scores = []
            for doc in all_docs:
                d_score = dense_norm.get(doc, 0.0)  # Missing doc gets 0.0
                s_score = sparse_norm.get(doc, 0.0)
                
                # Weighted formula
                f_score = (w * d_score) + ((1 - w) * s_score)
                final_scores.append((doc, f_score))
                
            # Sort by final fused score (highest first)
            final_scores.sort(key=lambda x: x[1], reverse=True)
            retrieved_doc_ids = [doc for doc, score in final_scores[:max(k_values)]]
            
            # Ground truth
            ground_truth = qrels[key]
            ground_truth_doc_id = str(ground_truth["section_id"]) + ground_truth["doc_id"]
            
            # Calculate Recall@K
            for k in k_values:
                retrieved_k = retrieved_doc_ids[:k]
                hit = int(ground_truth_doc_id in retrieved_k)
                metrics[f"Recall@{k}"].append(hit)

            # Calculate MRR
            rr = 0.0
            if ground_truth_doc_id in retrieved_doc_ids:
                rank = retrieved_doc_ids.index(ground_truth_doc_id) + 1
                rr = 1.0 / rank
            metrics["MRR"].append(rr)
            
        # Collect results for this weight
        final_metrics = {
            "Dense W": w,
            "Sparse W": round(1 - w, 1)
        }
        final_metrics.update({
            metric: round(np.mean(values), 4)
            for metric, values in metrics.items()
        })
        all_metrics.append(final_metrics)
        
    # 3. Present Results
    df = pd.DataFrame(all_metrics)
    
    # Identify the best weight based on MRR
    best_row = df.loc[df['MRR'].idxmax()]
    best_dense_w = best_row['Dense W']
    best_sparse_w = best_row['Sparse W']
    
    print(f"\n🏆 Best Configuration:")
    print(f"Highest MRR ({best_row['MRR']}) achieved with Dense Weight = {best_dense_w} and Sparse Weight = {best_sparse_w}")
    
    return df

# Run the evaluation
df_weighted_results = evaluate_weighted_fusion()
df_weighted_results.head()


Fetching initial results from Qdrant...
Fetched 50/334 queries...
Fetched 100/334 queries...
Fetched 150/334 queries...
Fetched 200/334 queries...
Fetched 250/334 queries...
Fetched 300/334 queries...

Evaluating weighted fusion...

🏆 Best Configuration:
Highest MRR (0.819) achieved with Dense Weight = 0.4 and Sparse Weight = 0.6


,Dense W,Sparse W,Recall@1,Recall@2,Recall@3,Recall@4,Recall@5,MRR
0,0.1,0.9,0.6737,0.8024,0.8892,0.9222,0.9461,0.7800
1,0.2,0.8,0.7006,0.8204,0.9012,0.9281,0.9551,0.7996
2,0.3,0.7,0.7096,0.8473,0.8952,0.9281,0.9521,0.8074
3,0.4,0.6,0.7275,0.8533,0.9102,0.9341,0.9521,0.8190
4,0.5,0.5,0.7126,0.8473,0.9102,0.9401,0.9521,0.8108


In [ ]:
df_weighted_results.to_csv("df_weighted_results.csv")

,Dense W,Sparse W,Recall@1,Recall@2,Recall@3,Recall@4,Recall@5,MRR
0,0.1,0.9,0.6737,0.8024,0.8892,0.9222,0.9461,0.7800
1,0.2,0.8,0.7006,0.8204,0.9012,0.9281,0.9551,0.7996
2,0.3,0.7,0.7096,0.8473,0.8952,0.9281,0.9521,0.8074
3,0.4,0.6,0.7275,0.8533,0.9102,0.9341,0.9521,0.8190
4,0.5,0.5,0.7126,0.8473,0.9102,0.9401,0.9521,0.8108
5,0.6,0.4,0.6766,0.8293,0.8982,0.9281,0.9521,0.7882
6,0.7,0.3,0.6407,0.8114,0.8832,0.9192,0.9431,0.7638
7,0.8,0.2,0.6168,0.7844,0.8563,0.9042,0.9251,0.7407
8,0.9,0.1,0.6168,0.7635,0.8263,0.8862,0.9072,0.7302


Use 3-4 Embedding Model 
- Thereare some retrival Specialized Embedding models

Write a blog on the retrival Methods

# Embedding model Evaluation

In [11]:
import numpy as np
import pandas as pd
from collections import defaultdict
from qdrant_client import QdrantClient, models

def evaluate_weighted_fusion():
    k_values = [1, 2, 3, 4, 5]
    embedding_model = SentenceTransformer("WhereIsAI/UAE-Large-V1", device = "cpu")
    
    # 1. Initialize Client
    client = QdrantClient(host="localhost", port=6333)
    bm25_model = SparseTextEmbedding(model_name="Qdrant/bm25")
    collection_name = "research_papers_v4_UAE"

    # Pre-fetch all query results to avoid redundant Qdrant calls across weights
    query_results = {}
    processed = 0
    
    print("Fetching initial results from Qdrant...")
    for key in queries_100:
        query = queries_100[key]["query"]

        dense_query_vector = embedding_model.encode(query)
        sparse_emb = list(bm25_model.embed([query]))[0]
        sparse_query_vector = models.SparseVector(
            indices=sparse_emb.indices.tolist(),
            values=sparse_emb.values.tolist()
        )

        # Get top 20 using dense embeddings
        dense_res = client.query_points(
            collection_name=collection_name,
            query=dense_query_vector,
            using="uae",
            limit=10,
            with_payload=True
        ).points
        
        # Get top 20 using sparse embeddings
        sparse_res = client.query_points(
            collection_name=collection_name,
            query=sparse_query_vector,
            using="bm25",
            limit=10,
            with_payload=True
        ).points
        
        query_results[key] = {
            "dense": dense_res,
            "sparse": sparse_res
        }
        
        processed += 1
        if processed % 50 == 0:
            print(f"Fetched {processed}/{len(queries_100)} queries...")

    # 2. Iterate over all weights [0.1, 0.2, 0.3, ... 0.9]
    weights = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    all_metrics = []

    print("\nEvaluating weighted fusion...")
    for w in weights:
        metrics = defaultdict(list)
        
        for key in queries_100:
            dense_res = query_results[key]["dense"]
            sparse_res = query_results[key]["sparse"]
            
            # Helper to apply Min-Max Normalization
            def process_results(res):
                scores = [p.score for p in res]
                if not scores:
                    return {}
                min_s, max_s = min(scores), max(scores)
                
                norm_dict = {}
                for p in res:
                    doc_id = str(p.payload.get("section_id", p.id) if p.payload else p.id)
                    # Normalize between 0 and 1
                    norm = (p.score - min_s) / (max_s - min_s) if max_s > min_s else 1.0
                    norm_dict[doc_id] = max(norm_dict.get(doc_id, 0), norm)
                return norm_dict
            
            dense_norm = process_results(dense_res)
            sparse_norm = process_results(sparse_res)
            
            # Union of all retrieved document IDs
            all_docs = set(dense_norm.keys()).union(set(sparse_norm.keys()))
            
            # Calculate final fused score
            final_scores = []
            for doc in all_docs:
                d_score = dense_norm.get(doc, 0.0)  # Missing doc gets 0.0
                s_score = sparse_norm.get(doc, 0.0)
                
                # Weighted formula
                f_score = (w * d_score) + ((1 - w) * s_score)
                final_scores.append((doc, f_score))
                
            # Sort by final fused score (highest first)
            final_scores.sort(key=lambda x: x[1], reverse=True)
            retrieved_doc_ids = [doc for doc, score in final_scores[:max(k_values)]]
            
            # Ground truth
            ground_truth = qrels[key]
            ground_truth_doc_id = str(ground_truth["section_id"]) + ground_truth["doc_id"]
            
            # Calculate Recall@K
            for k in k_values:
                retrieved_k = retrieved_doc_ids[:k]
                hit = int(ground_truth_doc_id in retrieved_k)
                metrics[f"Recall@{k}"].append(hit)

            # Calculate MRR
            rr = 0.0
            if ground_truth_doc_id in retrieved_doc_ids:
                rank = retrieved_doc_ids.index(ground_truth_doc_id) + 1
                rr = 1.0 / rank
            metrics["MRR"].append(rr)
            
        # Collect results for this weight
        final_metrics = {
            "Dense W": w,
            "Sparse W": round(1 - w, 1)
        }
        final_metrics.update({
            metric: round(np.mean(values), 4)
            for metric, values in metrics.items()
        })
        all_metrics.append(final_metrics)
        
    # 3. Present Results
    df = pd.DataFrame(all_metrics)
    
    # Identify the best weight based on MRR
    best_row = df.loc[df['MRR'].idxmax()]
    best_dense_w = best_row['Dense W']
    best_sparse_w = best_row['Sparse W']
    
    print(f"\n🏆 Best Configuration:")
    print(f"Highest MRR ({best_row['MRR']}) achieved with Dense Weight = {best_dense_w} and Sparse Weight = {best_sparse_w}")
    
    return df

# Run the evaluation
df_weighted_results_UAE = evaluate_weighted_fusion()
df_weighted_results_UAE.head()

Fetching initial results from Qdrant...
Fetched 50/334 queries...
Fetched 100/334 queries...
Fetched 150/334 queries...
Fetched 200/334 queries...
Fetched 250/334 queries...
Fetched 300/334 queries...

Evaluating weighted fusion...

🏆 Best Configuration:
Highest MRR (0.8223) achieved with Dense Weight = 0.4 and Sparse Weight = 0.6


,Dense W,Sparse W,Recall@1,Recall@2,Recall@3,Recall@4,Recall@5,MRR
0,0.1,0.9,0.7156,0.8293,0.9012,0.9371,0.9461,0.8072
1,0.2,0.8,0.7216,0.8383,0.9102,0.9341,0.9521,0.8135
2,0.3,0.7,0.7216,0.8533,0.9162,0.9431,0.9521,0.8169
3,0.4,0.6,0.7305,0.8503,0.9162,0.9461,0.9581,0.8223
4,0.5,0.5,0.7156,0.8323,0.9102,0.9461,0.9521,0.8101


In [12]:
df_weighted_results_UAE.to_csv("df_weighted_results_UAE.csv")

In [15]:
import numpy as np
import pandas as pd
from collections import defaultdict
from qdrant_client import QdrantClient, models

def evaluate_weighted_fusion():
    k_values = [1, 2, 3, 4, 5]    
    # 1. Initialize Client
    client = QdrantClient(host="localhost", port=6333)
    bm25_model = SparseTextEmbedding(model_name="Qdrant/bm25")
    collection_name = "research_papers_v4_bge"
    embedding_model = SentenceTransformer("BAAI/bge-large-en-v1.5", device="cpu")

    # Pre-fetch all query results to avoid redundant Qdrant calls across weights
    query_results = {}
    processed = 0
    
    print("Fetching initial results from Qdrant...")
    for key in queries_100:
        query = queries_100[key]["query"]

        dense_query_vector = embedding_model.encode(query)
        sparse_emb = list(bm25_model.embed([query]))[0]
        sparse_query_vector = models.SparseVector(
            indices=sparse_emb.indices.tolist(),
            values=sparse_emb.values.tolist()
        )

        # Get top 20 using dense embeddings
        dense_res = client.query_points(
            collection_name=collection_name,
            query=dense_query_vector,
            using="uae",
            limit=10,
            with_payload=True
        ).points
        
        # Get top 20 using sparse embeddings
        sparse_res = client.query_points(
            collection_name=collection_name,
            query=sparse_query_vector,
            using="bm25",
            limit=10,
            with_payload=True
        ).points
        
        query_results[key] = {
            "dense": dense_res,
            "sparse": sparse_res
        }
        
        processed += 1
        if processed % 50 == 0:
            print(f"Fetched {processed}/{len(queries_100)} queries...")

    # 2. Iterate over all weights [0.1, 0.2, 0.3, ... 0.9]
    weights = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    all_metrics = []

    print("\nEvaluating weighted fusion...")
    for w in weights:
        metrics = defaultdict(list)
        
        for key in queries_100:
            dense_res = query_results[key]["dense"]
            sparse_res = query_results[key]["sparse"]
            
            # Helper to apply Min-Max Normalization
            def process_results(res):
                scores = [p.score for p in res]
                if not scores:
                    return {}
                min_s, max_s = min(scores), max(scores)
                
                norm_dict = {}
                for p in res:
                    doc_id = str(p.payload.get("section_id", p.id) if p.payload else p.id)
                    # Normalize between 0 and 1
                    norm = (p.score - min_s) / (max_s - min_s) if max_s > min_s else 1.0
                    norm_dict[doc_id] = max(norm_dict.get(doc_id, 0), norm)
                return norm_dict
            
            dense_norm = process_results(dense_res)
            sparse_norm = process_results(sparse_res)
            
            # Union of all retrieved document IDs
            all_docs = set(dense_norm.keys()).union(set(sparse_norm.keys()))
            
            # Calculate final fused score
            final_scores = []
            for doc in all_docs:
                d_score = dense_norm.get(doc, 0.0)  # Missing doc gets 0.0
                s_score = sparse_norm.get(doc, 0.0)
                
                # Weighted formula
                f_score = (w * d_score) + ((1 - w) * s_score)
                final_scores.append((doc, f_score))
                
            # Sort by final fused score (highest first)
            final_scores.sort(key=lambda x: x[1], reverse=True)
            retrieved_doc_ids = [doc for doc, score in final_scores[:max(k_values)]]
            
            # Ground truth
            ground_truth = qrels[key]
            ground_truth_doc_id = str(ground_truth["section_id"]) + ground_truth["doc_id"]
            
            # Calculate Recall@K
            for k in k_values:
                retrieved_k = retrieved_doc_ids[:k]
                hit = int(ground_truth_doc_id in retrieved_k)
                metrics[f"Recall@{k}"].append(hit)

            # Calculate MRR
            rr = 0.0
            if ground_truth_doc_id in retrieved_doc_ids:
                rank = retrieved_doc_ids.index(ground_truth_doc_id) + 1
                rr = 1.0 / rank
            metrics["MRR"].append(rr)
            
        # Collect results for this weight
        final_metrics = {
            "Dense W": w,
            "Sparse W": round(1 - w, 1)
        }
        final_metrics.update({
            metric: round(np.mean(values), 4)
            for metric, values in metrics.items()
        })
        all_metrics.append(final_metrics)
        
    # 3. Present Results
    df = pd.DataFrame(all_metrics)
    
    # Identify the best weight based on MRR
    best_row = df.loc[df['MRR'].idxmax()]
    best_dense_w = best_row['Dense W']
    best_sparse_w = best_row['Sparse W']
    
    print(f"\n🏆 Best Configuration:")
    print(f"Highest MRR ({best_row['MRR']}) achieved with Dense Weight = {best_dense_w} and Sparse Weight = {best_sparse_w}")
    
    return df

# Run the evaluation
df_weighted_results_bge = evaluate_weighted_fusion()
df_weighted_results_bge.head()

Fetching initial results from Qdrant...
Fetched 50/334 queries...
Fetched 100/334 queries...
Fetched 150/334 queries...
Fetched 200/334 queries...
Fetched 250/334 queries...
Fetched 300/334 queries...

Evaluating weighted fusion...

🏆 Best Configuration:
Highest MRR (0.8137) achieved with Dense Weight = 0.2 and Sparse Weight = 0.8


,Dense W,Sparse W,Recall@1,Recall@2,Recall@3,Recall@4,Recall@5,MRR
0,0.1,0.9,0.7126,0.8263,0.9012,0.9371,0.9461,0.8052
1,0.2,0.8,0.7216,0.8323,0.9102,0.9341,0.9581,0.8137
2,0.3,0.7,0.7126,0.8413,0.9102,0.9401,0.9521,0.8098
3,0.4,0.6,0.7156,0.8533,0.9132,0.9401,0.9491,0.8129
4,0.5,0.5,0.6946,0.8323,0.9132,0.9401,0.9461,0.7984


In [16]:
df_weighted_results_bge.to_csv("df_weighted_results_bge.csv")


## Results

- BAAI/bge-large-en-v1.5 - "Best MRR **0.8137** at dense weight **0.2**
- mixedbread-ai/mxbai-embed-large-v1 - Best MRR **0.819** at **0.4** dense weight
- WhereIsAI/UAE-Large-V1 - Best MRR **0.8223** at **0.4** dense weight


### Conclusion 
- Bge is not giving very good results but mxbai and embed are giving better results.
- Hence we can choose a embedding model between UAE and mxbai, mxbai is smaller model and UAE give slightly higher results.

# MultiMoldal RAG

## Image and text embeddings in the same vectorspace

In [14]:
import numpy as np
import pandas as pd
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from fastembed import SparseTextEmbedding
from qdrant_client import QdrantClient, models

def evaluate_hybrid_retrieval():
    k_values = [1, 2, 3, 4, 5]
    metrics = defaultdict(list)
    total_images_retrieved = 0

    # 1. Initialize Clients and Models
    client = QdrantClient(host="localhost", port=6333)
    collection_name = "research_papers_v2_hybrid"

    print("Loading models...")
    bm25_model = SparseTextEmbedding(model_name="Qdrant/bm25")
    text_embedding_model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1", device="cpu")
    clip_model = SentenceTransformer("clip-ViT-B-32", device="cpu")

    processed = 0

    for key in queries_100:
        query = queries_100[key]["query"]

        # 2. Generate Vectors
        
        # A) Dense Text Vector (1024 dims)
        dense_query_vector = text_embedding_model.encode(query).tolist()
        
        # B) Sparse BM25 Vector
        sparse_emb = list(bm25_model.embed([query]))[0]
        sparse_query_vector = models.SparseVector(
            indices=sparse_emb.indices.tolist(),
            values=sparse_emb.values.tolist()
        )
        
        # C) CLIP Image Vector (padded to 1024 dims)
        # We must pad it because the 'dense' space in your DB was created with size 1024
        raw_clip_vector = clip_model.encode(query)
        padded_clip_vector = np.pad(raw_clip_vector, (0, 1024 - len(raw_clip_vector)), 'constant').tolist()

        # 3. Hybrid Search using RRF (3 Streams!)
        search_results = client.query_points(
            collection_name=collection_name,
            prefetch=[
                # Fetch Top 10 Text Chunks using mxbai Text Embeddings
                models.Prefetch(
                    query=dense_query_vector,
                    using="dense",
                    limit=10,
                ),
                # Fetch Top 10 Text Chunks using Sparse BM25 Embeddings
                models.Prefetch(
                    query=sparse_query_vector,
                    using="sparse",
                    limit=10,
                ),
                # Fetch Top 10 Images using Padded CLIP Embeddings
                # Notice we query the SAME "dense" space, but with our padded CLIP query!
                models.Prefetch(
                    query=padded_clip_vector,
                    using="dense",
                    limit=10,
                ),
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=5,
            with_payload=True,
            with_vectors=False
        ).points

        retrieved_doc_ids = []

        # Extract doc_ids and check for images
        for point in search_results:
            doc_id = point.payload.get("section_id") if point.payload else None
            if not doc_id:
                doc_id = point.id
            retrieved_doc_ids.append(str(doc_id))
            
            # FIXED INDENTATION: Check if the point is an image inside the loop
            is_image = point.payload.get("image_path", None) if point.payload else None
            if is_image:
                total_images_retrieved += 1
        
        # Ground Truth Evaluation
        ground_truth = qrels[key]
        ground_truth_doc_id = str(ground_truth["section_id"]) + ground_truth["doc_id"]

        # Recall@K
        for k in k_values:
            retrieved_k = retrieved_doc_ids[:k]
            hit = int(ground_truth_doc_id in retrieved_k)
            metrics[f"Recall@{k}"].append(hit)

        # Standard MRR
        rr = 0.0
        if ground_truth_doc_id in retrieved_doc_ids:
            rank = retrieved_doc_ids.index(ground_truth_doc_id) + 1
            rr = 1.0 / rank

        metrics["MRR"].append(rr)

        processed += 1
        if processed % 20 == 0:
            print(f"Processed {processed}/{len(queries_100)} queries...")

    # Calculate final means
    final_metrics = {
        metric: round(np.mean(values), 4)
        for metric, values in metrics.items()
    }

    return total_images_retrieved, final_metrics

# Run the evaluation
image_count, metrics = evaluate_hybrid_retrieval()

print("\n--- RESULTS ---")
print(f"Total Images in Top 5 (across all queries): {image_count}")
print("Metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v}")


Loading models...
Processed 20/334 queries...
Processed 40/334 queries...
Processed 60/334 queries...
Processed 80/334 queries...
Processed 100/334 queries...
Processed 120/334 queries...
Processed 140/334 queries...
Processed 160/334 queries...
Processed 180/334 queries...
Processed 200/334 queries...
Processed 220/334 queries...
Processed 240/334 queries...
Processed 260/334 queries...
Processed 280/334 queries...
Processed 300/334 queries...
Processed 320/334 queries...

--- RESULTS ---
Total Images in Top 5 (across all queries): 460
Metrics:
  Recall@1: 0.7186
  Recall@2: 0.8024
  Recall@3: 0.8473
  Recall@4: 0.8982
  Recall@5: 0.9132
  MRR: 0.7912


## Image and text embeddings in different vectorspace 

In [5]:
import numpy as np
from collections import defaultdict
from qdrant_client import QdrantClient, models
from fastembed import SparseTextEmbedding
from sentence_transformers import SentenceTransformer

def compute_manual_rrf(list1, list2, k=60):
    """
    Computes Rank Reciprocal Fusion (RRF) across two lists of Qdrant points.
    Returns a single sorted list of points.
    """
    rrf_scores = defaultdict(float)
    point_map = {}
    
    for rank, point in enumerate(list1):
        rrf_scores[point.id] += 1.0 / (k + rank + 1)
        point_map[point.id] = point
        
    for rank, point in enumerate(list2):
        rrf_scores[point.id] += 1.0 / (k + rank + 1)
        point_map[point.id] = point
        
    # Sort by RRF score descending
    sorted_ids = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)
    return [point_map[pid] for pid in sorted_ids]


def evaluate_dual_hybrid_retrieval():
    k_values = [1, 2, 3, 4, 5]
    metrics = defaultdict(list)
    total_images_retrieved = 0

    # 1. Initialize Clients and Models
    client = QdrantClient(host="localhost", port=6333)
    
    TEXT_COLLECTION = "research_papers_text_v5"
    IMAGE_COLLECTION = "research_papers_image_v5"

    print("Loading models...")
    bm25_model = SparseTextEmbedding(model_name="Qdrant/bm25")
    embedding_model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1", device="cpu")
    clip_model = SentenceTransformer('clip-ViT-B-32', device='cpu')

    processed = 0

    for key in queries_100:
        query = queries_100[key]["query"]
        
        # Determine if query type is strictly text or involves images
        query_type = queries_100[key].get("source", "text")

        # 2. Generate Vectors for Text Collection
        dense_query_vector = embedding_model.encode(query)
        
        sparse_emb = list(bm25_model.embed([query]))[0]
        sparse_query_vector = models.SparseVector(
            indices=sparse_emb.indices.tolist(),
            values=sparse_emb.values.tolist()
        )

        # 3. Search TEXT collection (Hybrid with Qdrant RRF)
        text_search_results = client.query_points(
            collection_name=TEXT_COLLECTION,
            prefetch=[
                models.Prefetch(
                    query=dense_query_vector.tolist(),
                    using="dense",
                    limit=10, 
                ),
                models.Prefetch(
                    query=sparse_query_vector,
                    using="sparse",
                    limit=10, 
                ),
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=5, 
            with_payload=True,
            with_vectors=False
        ).points
        
        final_search_results = text_search_results

        # 4. If query needs image vectorstore, query it and combine with manual RRF
        if query_type != "text":
            image_query_vector = clip_model.encode(query)
            
            # Fetch top 10 from Image vectorstore 
            # REMOVED using="dense" because image collection uses the default unnamed vector
            image_search_results = client.query_points(
                collection_name=IMAGE_COLLECTION,
                query=image_query_vector.tolist(),
                limit=10,
                with_payload=True,
                with_vectors=False
            ).points
            
            # Combine text results and image results using manual RRF
            combined_results = compute_manual_rrf(text_search_results, image_search_results)
            
            # Keep top 5 from the fused results
            final_search_results = combined_results[:5]
            
        # 5. Extract results (No Reranking step!)
        retrieved_doc_ids = []

        # Extract doc_ids as before
        for point in final_search_results:
            doc_id = point.payload.get("section_id") if point.payload else None
            if not doc_id:
                doc_id = point.id
            retrieved_doc_ids.append(str(doc_id))
            
            # Check if the point is an image and update the counter
            is_image = point.payload.get("image_path", None) if point.payload else None
            if is_image:
                total_images_retrieved += 1

        # Ground Truth Evaluation
        ground_truth = qrels[key]
        ground_truth_doc_id = str(ground_truth["section_id"]) + ground_truth["doc_id"]

        # Recall@K
        for k in k_values:
            retrieved_k = retrieved_doc_ids[:k]
            hit = int(ground_truth_doc_id in retrieved_k)
            metrics[f"Recall@{k}"].append(hit)

        # Standard MRR
        rr = 0.0
        if ground_truth_doc_id in retrieved_doc_ids:
            rank = retrieved_doc_ids.index(ground_truth_doc_id) + 1
            rr = 1.0 / rank

        metrics["MRR"].append(rr)

        processed += 1
        if processed % 20 == 0:
            print(f"Processed {processed}/{len(queries_100)} queries...")

    # Calculate final means
    final_metrics = {
        metric: round(np.mean(values), 4)
        for metric, values in metrics.items()
    }

    return total_images_retrieved, final_metrics

# Run the evaluation
image_count, dual_hybrid_metrics = evaluate_dual_hybrid_retrieval()

print("\n--- RESULTS ---")
print(f"Total Images in Top 5 (across all queries): {image_count}")
print("Dual Hybrid Search Results:", dual_hybrid_metrics)


Loading models...
Processed 20/334 queries...
Processed 40/334 queries...
Processed 60/334 queries...
Processed 80/334 queries...
Processed 100/334 queries...
Processed 120/334 queries...
Processed 140/334 queries...
Processed 160/334 queries...
Processed 180/334 queries...
Processed 200/334 queries...
Processed 220/334 queries...
Processed 240/334 queries...
Processed 260/334 queries...
Processed 280/334 queries...
Processed 300/334 queries...
Processed 320/334 queries...

--- RESULTS ---
Total Images in Top 5 (across all queries): 224
Dual Hybrid Search Results: {'Recall@1': np.float64(0.7246), 'Recall@2': np.float64(0.7994), 'Recall@3': np.float64(0.8713), 'Recall@4': np.float64(0.8982), 'Recall@5': np.float64(0.9251), 'MRR': np.float64(0.7981)}


In [5]:
import numpy as np
from collections import defaultdict
from qdrant_client import QdrantClient, models
from fastembed import SparseTextEmbedding
from sentence_transformers import SentenceTransformer

def compute_manual_rrf(list1, list2, k=60):
    """
    Computes Rank Reciprocal Fusion (RRF) across two lists of Qdrant points.
    Returns a single sorted list of points.
    """
    rrf_scores = defaultdict(float)
    point_map = {}
    
    for rank, point in enumerate(list1):
        rrf_scores[point.id] += 1.0 / (k + rank + 1)
        point_map[point.id] = point
        
    for rank, point in enumerate(list2):
        rrf_scores[point.id] += 1.0 / (k + rank + 1)
        point_map[point.id] = point
        
    # Sort by RRF score descending
    sorted_ids = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)
    return [point_map[pid] for pid in sorted_ids]


def compute_weighted_fusion(dense_list, sparse_list, alpha=0.4):
    """
    Computes Weighted Fusion across two lists of Qdrant points.
    Alpha controls the weight of the dense_list, and (1-alpha) controls sparse_list.
    Returns a single sorted list of points.
    """
    dense_scores = {p.id: p.score for p in dense_list}
    sparse_scores = {p.id: p.score for p in sparse_list}
    
    # Normalize scores (Min-Max scaling)
    def normalize(scores):
        if not scores:
            return scores
        min_val = min(scores.values())
        max_val = max(scores.values())
        if max_val == min_val:
            return {k: 1.0 for k in scores}
        return {k: (v - min_val) / (max_val - min_val) for k, v in scores.items()}
        
    norm_dense = normalize(dense_scores)
    norm_sparse = normalize(sparse_scores)
    
    fused_scores = {}
    point_map = {}
    
    for p in dense_list:
        point_map[p.id] = p
        fused_scores[p.id] = alpha * norm_dense.get(p.id, 0)
        
    for p in sparse_list:
        point_map[p.id] = p
        if p.id in fused_scores:
            fused_scores[p.id] += (1 - alpha) * norm_sparse.get(p.id, 0)
        else:
            fused_scores[p.id] = (1 - alpha) * norm_sparse.get(p.id, 0)
            
    # Sort by fused score descending
    sorted_ids = sorted(fused_scores.keys(), key=lambda x: fused_scores[x], reverse=True)
    return [point_map[pid] for pid in sorted_ids]


def evaluate_dual_hybrid_retrieval():
    k_values = [1, 2, 3, 4, 5]
    metrics = defaultdict(list)
    total_images_retrieved = 0

    # 1. Initialize Clients and Models
    client = QdrantClient(host="localhost", port=6333)
    
    TEXT_COLLECTION = "research_papers_text_v5"
    IMAGE_COLLECTION = "research_papers_image_v5"

    print("Loading models...")
    bm25_model = SparseTextEmbedding(model_name="Qdrant/bm25")
    embedding_model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1", device="cpu")
    clip_model = SentenceTransformer('clip-ViT-B-32', device='cpu')

    processed = 0

    for key in queries_100:
        query = queries_100[key]["query"]
        
        # Determine if query type is strictly text or involves images
        query_type = queries_100[key].get("source", "text")

        # 2. Generate Vectors for Text Collection
        dense_query_vector = embedding_model.encode(query)
        
        sparse_emb = list(bm25_model.embed([query]))[0]
        sparse_query_vector = models.SparseVector(
            indices=sparse_emb.indices.tolist(),
            values=sparse_emb.values.tolist()
        )

        # 3. Search TEXT collection (Hybrid with Weighted Fusion)
        dense_results = client.query_points(
            collection_name=TEXT_COLLECTION,
            query=dense_query_vector.tolist(),
            using="dense",
            limit=10, 
            with_payload=True,
            with_vectors=False
        ).points
        
        sparse_results = client.query_points(
            collection_name=TEXT_COLLECTION,
            query=sparse_query_vector,
            using="sparse",
            limit=10, 
            with_payload=True,
            with_vectors=False
        ).points
        
        # Combine dense and sparse results using manual weighted fusion (alpha=0.4)
        text_search_results = compute_weighted_fusion(dense_results, sparse_results, alpha=0.4)[:5]
        
        final_search_results = text_search_results

        # 4. If query needs image vectorstore, query it and combine with manual RRF
        if query_type != "text":
            image_query_vector = clip_model.encode(query)
            
            # Fetch top 10 from Image vectorstore 
            # REMOVED using="dense" because image collection uses the default unnamed vector
            image_search_results = client.query_points(
                collection_name=IMAGE_COLLECTION,
                query=image_query_vector.tolist(),
                limit=10,
                with_payload=True,
                with_vectors=False
            ).points
            
            # Combine text results and image results using manual RRF
            combined_results = compute_manual_rrf(text_search_results, image_search_results)
            
            # Keep top 5 from the fused results
            final_search_results = combined_results[:5]
            
        # 5. Extract results (No Reranking step!)
        retrieved_doc_ids = []

        # Extract doc_ids as before
        for point in final_search_results:
            doc_id = point.payload.get("section_id") if point.payload else None
            if not doc_id:
                doc_id = point.id
            retrieved_doc_ids.append(str(doc_id))
            
            # Check if the point is an image and update the counter
            is_image = point.payload.get("image_path", None) if point.payload else None
            if is_image:
                total_images_retrieved += 1

        # Ground Truth Evaluation
        ground_truth = qrels[key]
        ground_truth_doc_id = str(ground_truth["section_id"]) + ground_truth["doc_id"]

        # Recall@K
        for k in k_values:
            retrieved_k = retrieved_doc_ids[:k]
            hit = int(ground_truth_doc_id in retrieved_k)
            metrics[f"Recall@{k}"].append(hit)

        # Standard MRR
        rr = 0.0
        if ground_truth_doc_id in retrieved_doc_ids:
            rank = retrieved_doc_ids.index(ground_truth_doc_id) + 1
            rr = 1.0 / rank

        metrics["MRR"].append(rr)

        processed += 1
        if processed % 20 == 0:
            print(f"Processed {processed}/{len(queries_100)} queries...")

    # Calculate final means
    final_metrics = {
        metric: round(np.mean(values), 4)
        for metric, values in metrics.items()
    }

    return total_images_retrieved, final_metrics

# Run the evaluation
image_count, dual_hybrid_metrics = evaluate_dual_hybrid_retrieval()

print("\n--- RESULTS ---")
print(f"Total Images in Top 5 (across all queries): {image_count}")
print("Dual Hybrid Search Results:", dual_hybrid_metrics)


Loading models...
Processed 20/334 queries...
Processed 40/334 queries...
Processed 60/334 queries...
Processed 80/334 queries...
Processed 100/334 queries...
Processed 120/334 queries...
Processed 140/334 queries...
Processed 160/334 queries...
Processed 180/334 queries...
Processed 200/334 queries...
Processed 220/334 queries...
Processed 240/334 queries...
Processed 260/334 queries...
Processed 280/334 queries...
Processed 300/334 queries...
Processed 320/334 queries...

--- RESULTS ---
Total Images in Top 5 (across all queries): 224
Dual Hybrid Search Results: {'Recall@1': np.float64(0.7186), 'Recall@2': np.float64(0.7964), 'Recall@3': np.float64(0.8713), 'Recall@4': np.float64(0.8922), 'Recall@5': np.float64(0.9192), 'MRR': np.float64(0.7931)}


Dual Hybrid Search Results: 

- 'Recall@1': np.float64(0.7246), 
- 'Recall@2': np.float64(0.7994), 
- 'Recall@3': np.float64(0.8713), 
- 'Recall@4': np.float64(0.8982), 
- 'Recall@5': np.float64(0.9251), 
- 'MRR': np.float64(0.7981)

Same VectoreSpace

Metrics:
- Recall@1: 0.7186
-  Recall@2: 0.8024
-  Recall@3: 0.8473
-  Recall@4: 0.8982
-  Recall@5: 0.9132
-  MRR: 0.7912


In [6]:
import numpy as np
import pandas as pd
from collections import defaultdict
from qdrant_client import QdrantClient, models
from fastembed import SparseTextEmbedding
from sentence_transformers import SentenceTransformer


def compute_weighted_fusion(list1, list2, alpha=0.4):
    """
    Computes Weighted Fusion across two lists of Qdrant points.
    Alpha controls the weight of list1, and (1-alpha) controls list2.
    Returns a single sorted list of points.
    """
    scores_1 = {p.id: p.score for p in list1}
    scores_2 = {p.id: p.score for p in list2}
    
    # Normalize scores (Min-Max scaling)
    def normalize(scores):
        if not scores:
            return scores
        min_val = min(scores.values())
        max_val = max(scores.values())
        if max_val == min_val:
            return {k: 1.0 for k in scores}
        return {k: (v - min_val) / (max_val - min_val) for k, v in scores.items()}
        
    norm_1 = normalize(scores_1)
    norm_2 = normalize(scores_2)
    
    fused_scores = {}
    point_map = {}
    
    for p in list1:
        point_map[p.id] = p
        fused_scores[p.id] = alpha * norm_1.get(p.id, 0)
        
    for p in list2:
        point_map[p.id] = p
        if p.id in fused_scores:
            fused_scores[p.id] += (1 - alpha) * norm_2.get(p.id, 0)
        else:
            fused_scores[p.id] = (1 - alpha) * norm_2.get(p.id, 0)
            
    # Sort by fused score descending
    sorted_ids = sorted(fused_scores.keys(), key=lambda x: fused_scores[x], reverse=True)
    return [point_map[pid] for pid in sorted_ids]


def evaluate_dual_hybrid_retrieval_sweep():
    k_values = [1, 2, 3, 4, 5]
    alpha_values = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    
    # Track metrics for each alpha
    all_metrics = {alpha: defaultdict(list) for alpha in alpha_values}
    total_images_retrieved = {alpha: 0 for alpha in alpha_values}

    # 1. Initialize Clients and Models
    client = QdrantClient(host="localhost", port=6333)
    
    TEXT_COLLECTION = "research_papers_text_v5"
    IMAGE_COLLECTION = "research_papers_image_v5"

    print("Loading models...")
    bm25_model = SparseTextEmbedding(model_name="Qdrant/bm25")
    embedding_model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1", device="cpu")
    clip_model = SentenceTransformer('clip-ViT-B-32', device='cpu')

    processed = 0

    for key in queries_100:
        query = queries_100[key]["query"]
        
        # Determine if query type is strictly text or involves images
        query_type = queries_100[key].get("source", "text")

        # 2. Generate Vectors for Text Collection
        dense_query_vector = embedding_model.encode(query)
        
        sparse_emb = list(bm25_model.embed([query]))[0]
        sparse_query_vector = models.SparseVector(
            indices=sparse_emb.indices.tolist(),
            values=sparse_emb.values.tolist()
        )

        # 3. Search TEXT collection (Hybrid with Weighted Fusion, fixed alpha=0.4)
        dense_results = client.query_points(
            collection_name=TEXT_COLLECTION,
            query=dense_query_vector.tolist(),
            using="dense",
            limit=10, 
            with_payload=True,
            with_vectors=False
        ).points
        
        sparse_results = client.query_points(
            collection_name=TEXT_COLLECTION,
            query=sparse_query_vector,
            using="sparse",
            limit=10, 
            with_payload=True,
            with_vectors=False
        ).points
        
        # Combine dense and sparse results using manual weighted fusion (alpha=0.4)
        text_search_results = compute_weighted_fusion(dense_results, sparse_results, alpha=0.4)[:5]
        
        final_search_results_by_alpha = {}

        # 4. If query needs image vectorstore, query it and combine with manual weighted fusion
        if query_type != "text":
            image_query_vector = clip_model.encode(query)
            
            # Fetch top 10 from Image vectorstore 
            image_search_results = client.query_points(
                collection_name=IMAGE_COLLECTION,
                query=image_query_vector.tolist(),
                limit=10,
                with_payload=True,
                with_vectors=False
            ).points
            
            # Combine text and image results using weighted fusion for each alpha
            for alpha in alpha_values:
                # alpha weight for text, (1-alpha) for image
                combined_results = compute_weighted_fusion(text_search_results, image_search_results, alpha=alpha)
                final_search_results_by_alpha[alpha] = combined_results[:5]
        else:
            # If purely text, the results are identical across all alphas
            for alpha in alpha_values:
                final_search_results_by_alpha[alpha] = text_search_results[:5]
            
        # 5. Extract results and evaluate for each alpha
        ground_truth = qrels[key]
        ground_truth_doc_id = str(ground_truth["section_id"]) + ground_truth["doc_id"]

        for alpha in alpha_values:
            retrieved_doc_ids = []

            for point in final_search_results_by_alpha[alpha]:
                doc_id = point.payload.get("section_id") if point.payload else None
                if not doc_id:
                    doc_id = point.id
                retrieved_doc_ids.append(str(doc_id))
                
                is_image = point.payload.get("image_path", None) if point.payload else None
                if is_image:
                    total_images_retrieved[alpha] += 1

            # Recall@K
            for k in k_values:
                retrieved_k = retrieved_doc_ids[:k]
                hit = int(ground_truth_doc_id in retrieved_k)
                all_metrics[alpha][f"Recall@{k}"].append(hit)

            # Standard MRR
            rr = 0.0
            if ground_truth_doc_id in retrieved_doc_ids:
                rank = retrieved_doc_ids.index(ground_truth_doc_id) + 1
                rr = 1.0 / rank

            all_metrics[alpha]["MRR"].append(rr)

        processed += 1
        if processed % 20 == 0:
            print(f"Processed {processed}/{len(queries_100)} queries...")

    # Calculate final means and compile into a DataFrame
    results_list = []
    for alpha in alpha_values:
        alpha_summary = {"Alpha (Text Weight)": alpha}
        for metric, values in all_metrics[alpha].items():
            alpha_summary[metric] = round(np.mean(values), 4)
        alpha_summary["Images_in_Top5"] = total_images_retrieved[alpha]
        results_list.append(alpha_summary)

    df_results = pd.DataFrame(results_list)
    return df_results

# Run the evaluation and display the dataframe
df_metrics = evaluate_dual_hybrid_retrieval_sweep()
print("\n--- RESULTS ---")
display(df_metrics)


Loading models...
Processed 20/334 queries...
Processed 40/334 queries...
Processed 60/334 queries...
Processed 80/334 queries...
Processed 100/334 queries...
Processed 120/334 queries...
Processed 140/334 queries...
Processed 160/334 queries...
Processed 180/334 queries...
Processed 200/334 queries...
Processed 220/334 queries...
Processed 240/334 queries...
Processed 260/334 queries...
Processed 280/334 queries...
Processed 300/334 queries...
Processed 320/334 queries...

--- RESULTS ---


,Alpha (Text Weight),Recall@1,Recall@2,Recall@3,Recall@4,Recall@5,MRR,Images_in_Top5
0,0.1,0.5389,0.6228,0.6677,0.6886,0.7066,0.6046,552
1,0.2,0.5389,0.6228,0.6796,0.7156,0.7725,0.6202,494
2,0.3,0.5389,0.6287,0.7066,0.7754,0.8353,0.6390,420
3,0.4,0.5389,0.6617,0.7784,0.8443,0.8832,0.6635,339
4,0.5,0.6886,0.7695,0.8443,0.8713,0.8982,0.7661,269
5,0.6,0.6886,0.8024,0.8593,0.8892,0.9012,0.7744,224
6,0.7,0.6886,0.8024,0.8653,0.8952,0.9102,0.7769,192
7,0.8,0.6886,0.8024,0.8683,0.8982,0.9162,0.7785,172
8,0.9,0.6886,0.8024,0.8713,0.9042,0.9192,0.7797,155


In [7]:
df_metrics.head()

,Alpha (Text Weight),Recall@1,Recall@2,Recall@3,Recall@4,Recall@5,MRR,Images_in_Top5
0,0.1,0.5389,0.6228,0.6677,0.6886,0.7066,0.6046,552
1,0.2,0.5389,0.6228,0.6796,0.7156,0.7725,0.6202,494
2,0.3,0.5389,0.6287,0.7066,0.7754,0.8353,0.6390,420
3,0.4,0.5389,0.6617,0.7784,0.8443,0.8832,0.6635,339
4,0.5,0.6886,0.7695,0.8443,0.8713,0.8982,0.7661,269


# Using Different Image Embedding models


### Using SGlip

In [2]:
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModel

class SiglipEmbedder:
    def __init__(self, model_name="google/siglip-base-patch16-512", device="cpu"):
        print(f"Loading {model_name}...")
        self.processor = AutoProcessor.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(device)
        self.device = device
        
    def encode(self, text):
        """Encodes text into a normalized SigLIP vector."""
        if isinstance(text, str):
            text = [text]
            
        inputs = self.processor(text=text, padding="max_length", return_tensors="pt").to(self.device)
        with torch.no_grad():
            text_features = self.model.get_text_features(**inputs)
            # SigLIP requires manual L2 normalization for cosine similarity
            text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)
            
        if len(text) == 1:
            return text_features[0].cpu().numpy()
        return text_features.cpu().numpy()

    def encode_image(self, image):
        """Helper function if you need to re-index your images into Qdrant."""
        # If a file path is provided as a string, open it with PIL
        if isinstance(image, str):
            image = Image.open(image).convert("RGB")
            
        inputs = self.processor(images=image, return_tensors="pt").to(self.device)
        with torch.no_grad():
            image_features = self.model.get_image_features(**inputs)
            image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
            
        return image_features[0].cpu().numpy()


In [6]:
import numpy as np
import pandas as pd
from collections import defaultdict
from qdrant_client import QdrantClient, models
from fastembed import SparseTextEmbedding
from sentence_transformers import SentenceTransformer


def compute_weighted_fusion(list1, list2, alpha=0.4):
    """
    Computes Weighted Fusion across two lists of Qdrant points.
    Alpha controls the weight of list1, and (1-alpha) controls list2.
    Returns a single sorted list of points.
    """
    scores_1 = {p.id: p.score for p in list1}
    scores_2 = {p.id: p.score for p in list2}
    
    # Normalize scores (Min-Max scaling)
    def normalize(scores):
        if not scores:
            return scores
        min_val = min(scores.values())
        max_val = max(scores.values())
        if max_val == min_val:
            return {k: 1.0 for k in scores}
        return {k: (v - min_val) / (max_val - min_val) for k, v in scores.items()}
        
    norm_1 = normalize(scores_1)
    norm_2 = normalize(scores_2)
    
    fused_scores = {}
    point_map = {}
    
    for p in list1:
        point_map[p.id] = p
        fused_scores[p.id] = alpha * norm_1.get(p.id, 0)
        
    for p in list2:
        point_map[p.id] = p
        if p.id in fused_scores:
            fused_scores[p.id] += (1 - alpha) * norm_2.get(p.id, 0)
        else:
            fused_scores[p.id] = (1 - alpha) * norm_2.get(p.id, 0)
            
    # Sort by fused score descending
    sorted_ids = sorted(fused_scores.keys(), key=lambda x: fused_scores[x], reverse=True)
    return [point_map[pid] for pid in sorted_ids]


def evaluate_dual_hybrid_retrieval_sweep():
    k_values = [1, 2, 3, 4, 5]
    alpha_values = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    
    # Track metrics for each alpha
    all_metrics = {alpha: defaultdict(list) for alpha in alpha_values}
    total_images_retrieved = {alpha: 0 for alpha in alpha_values}

    # 1. Initialize Clients and Models
    client = QdrantClient(host="localhost", port=6333)
    
    TEXT_COLLECTION = "research_papers_text_v5"
    IMAGE_COLLECTION = "research_papers_image_v6"

    print("Loading models...")
    bm25_model = SparseTextEmbedding(model_name="Qdrant/bm25")
    embedding_model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1", device="cpu")
    # clip_model = SentenceTransformer('clip-ViT-B-32', device='cpu')
    siglip_model = SiglipEmbedder("google/siglip-base-patch16-512", device="cpu") # switch to "cuda" if you have it!

    processed = 0

    for key in queries_100:
        query = queries_100[key]["query"]
        
        # Determine if query type is strictly text or involves images
        query_type = queries_100[key].get("source", "text")

        # 2. Generate Vectors for Text Collection
        dense_query_vector = embedding_model.encode(query)
        
        sparse_emb = list(bm25_model.embed([query]))[0]
        sparse_query_vector = models.SparseVector(
            indices=sparse_emb.indices.tolist(),
            values=sparse_emb.values.tolist()
        )

        # 3. Search TEXT collection (Hybrid with Weighted Fusion, fixed alpha=0.4)
        dense_results = client.query_points(
            collection_name=TEXT_COLLECTION,
            query=dense_query_vector.tolist(),
            using="dense",
            limit=10, 
            with_payload=True,
            with_vectors=False
        ).points
        
        sparse_results = client.query_points(
            collection_name=TEXT_COLLECTION,
            query=sparse_query_vector,
            using="sparse",
            limit=10, 
            with_payload=True,
            with_vectors=False
        ).points
        
        # Combine dense and sparse results using manual weighted fusion (alpha=0.4)
        text_search_results = compute_weighted_fusion(dense_results, sparse_results, alpha=0.4)[:5]
        
        final_search_results_by_alpha = {}

        # 4. If query needs image vectorstore, query it and combine with manual weighted fusion
        if query_type != "text":
            image_query_vector = siglip_model.encode(query)
            
            # Fetch top 10 from Image vectorstore 
            image_search_results = client.query_points(
                collection_name=IMAGE_COLLECTION,
                query=image_query_vector.tolist(),
                limit=10,
                with_payload=True,
                with_vectors=False
            ).points
            
            # Combine text and image results using weighted fusion for each alpha
            for alpha in alpha_values:
                # alpha weight for text, (1-alpha) for image
                combined_results = compute_weighted_fusion(text_search_results, image_search_results, alpha=alpha)
                final_search_results_by_alpha[alpha] = combined_results[:5]
        else:
            # If purely text, the results are identical across all alphas
            for alpha in alpha_values:
                final_search_results_by_alpha[alpha] = text_search_results[:5]
            
        # 5. Extract results and evaluate for each alpha
        ground_truth = qrels[key]
        ground_truth_doc_id = str(ground_truth["section_id"]) + ground_truth["doc_id"]

        for alpha in alpha_values:
            retrieved_doc_ids = []

            for point in final_search_results_by_alpha[alpha]:
                doc_id = point.payload.get("section_id") if point.payload else None
                if not doc_id:
                    doc_id = point.id
                retrieved_doc_ids.append(str(doc_id))
                
                is_image = point.payload.get("image_path", None) if point.payload else None
                if is_image:
                    total_images_retrieved[alpha] += 1

            # Recall@K
            for k in k_values:
                retrieved_k = retrieved_doc_ids[:k]
                hit = int(ground_truth_doc_id in retrieved_k)
                all_metrics[alpha][f"Recall@{k}"].append(hit)

            # Standard MRR
            rr = 0.0
            if ground_truth_doc_id in retrieved_doc_ids:
                rank = retrieved_doc_ids.index(ground_truth_doc_id) + 1
                rr = 1.0 / rank

            all_metrics[alpha]["MRR"].append(rr)

        processed += 1
        if processed % 20 == 0:
            print(f"Processed {processed}/{len(queries_100)} queries...")

    # Calculate final means and compile into a DataFrame
    results_list = []
    for alpha in alpha_values:
        alpha_summary = {"Alpha (Text Weight)": alpha}
        for metric, values in all_metrics[alpha].items():
            alpha_summary[metric] = round(np.mean(values), 4)
        alpha_summary["Images_in_Top5"] = total_images_retrieved[alpha]
        results_list.append(alpha_summary)

    df_results = pd.DataFrame(results_list)
    return df_results

# Run the evaluation and display the dataframe
df_metrics = evaluate_dual_hybrid_retrieval_sweep()
print("\n--- RESULTS ---")
display(df_metrics)


Loading models...
Loading google/siglip-base-patch16-512...
Processed 20/334 queries...
Processed 40/334 queries...
Processed 60/334 queries...
Processed 80/334 queries...
Processed 100/334 queries...
Processed 120/334 queries...
Processed 140/334 queries...
Processed 160/334 queries...
Processed 180/334 queries...
Processed 200/334 queries...
Processed 220/334 queries...
Processed 240/334 queries...
Processed 260/334 queries...
Processed 280/334 queries...
Processed 300/334 queries...
Processed 320/334 queries...

--- RESULTS ---


,Alpha (Text Weight),Recall@1,Recall@2,Recall@3,Recall@4,Recall@5,MRR,Images_in_Top5
0,0.1,0.5778,0.6647,0.7126,0.7365,0.7665,0.6492,548
1,0.2,0.5778,0.6647,0.7126,0.7635,0.8114,0.6595,486
2,0.3,0.5778,0.6766,0.7335,0.8054,0.8503,0.6732,422
3,0.4,0.5778,0.7006,0.7695,0.8413,0.8802,0.6879,349
4,0.5,0.6886,0.7754,0.8473,0.8772,0.8982,0.7677,268
5,0.6,0.6886,0.7994,0.8593,0.8862,0.9072,0.7749,223
6,0.7,0.6886,0.8024,0.8623,0.8892,0.9072,0.7758,198
7,0.8,0.6886,0.8024,0.8683,0.8922,0.9162,0.7782,171
8,0.9,0.6886,0.8024,0.8713,0.9012,0.9192,0.7795,156


## Results

- The image is retrival is not getting the best image. But we can still fetch the images seperately from the image vectorstore and give it to the LLM.And see the results of the LLM

- For now the lesses the focus is on images better the accuracy of retrival.

In [10]:
ima_queries = 0
for i in queries_100:
    if queries_100[i]['source'] == "text-image":
        ima_queries+=1

In [11]:
ima_queries

74

In [12]:
len(queries_100)

334

In [1]:
import os
import base64
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.messages import HumanMessage

# 1. Load the API key from your local .env file
load_dotenv("/home/jugal/Documents/DataScience/Advanced_RAG/.env")

if "NVIDIA_API_KEY" not in os.environ:
    raise ValueError("⚠️ NVIDIA_API_KEY not found in .env file! Please check the file path.")

# 2. Initialize the Llama 3.1 Nemotron Nano VL 8B v1 Vision model
print("Initializing model: nvidia/llama-3.1-nemotron-nano-vl-8b-v1")
vision_model = ChatNVIDIA(model="nvidia/llama-3.1-nemotron-nano-vl-8b-v1")

# 3. Helper function to encode image to base64
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# Choose a test image from your directory
image_path = "/home/jugal/Documents/DataScience/Advanced_RAG/Research_Paper_rag/img7.jpg"

# 4. Convert the image to a base64 data URI (required by LangChain for images)
print(f"Encoding image: {image_path}")
image_b64 = encode_image(image_path)

# Assuming it's a JPEG. Change the mime type to image/png if it's a PNG
image_data_uri = f"data:image/jpeg;base64,{image_b64}"

# 5. Construct the multimodal prompt
message = HumanMessage(
    content=[
        {
            "type": "text", 
            "text": "Analyze this research paper image. Describe any graphs, charts, diagrams, or text you can see in detail."
        },
        {
            "type": "image_url", 
            "image_url": {"url": image_data_uri}
        }
    ]
)

# 6. Call the NVIDIA API!
print("Sending request to NVIDIA NIM API...")
response = vision_model.invoke([message])

print("\n--- Response from Llama 3.1 Nemotron Nano VL ---")
print(response.content)


/home/jugal/miniconda3/envs/ds/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Initializing model: nvidia/llama-3.1-nemotron-nano-vl-8b-v1
Encoding image: /home/jugal/Documents/DataScience/Advanced_RAG/Research_Paper_rag/img7.jpg
Sending request to NVIDIA NIM API...

--- Response from Llama 3.1 Nemotron Nano VL ---
The image is a scatter plot graph that depicts the relationship between GOR and NER. The graph is on a black and white background with a scale of 0 to 100 on the x-axis. The y-axis contains values from 0 to 100. The line of best fit moves diagonally from the lower left corner to the upper right corner. There is a black line of best fit and another black dotted line below it. The graph has a cloud-like appearance because the shadow is made by the black dots representing the data. Below the black line of best fit is a straight black line that runs from the top left corner to the lower right corner.


In [2]:
import os
import base64
import torch
from dotenv import load_dotenv
from PIL import Image

from qdrant_client import QdrantClient, models
from fastembed import SparseTextEmbedding
from sentence_transformers import SentenceTransformer
from transformers import AutoProcessor, AutoModel

from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.messages import HumanMessage

# ==========================================
# 1. Initialization & Setup
# ==========================================
load_dotenv("/home/jugal/Documents/DataScience/Advanced_RAG/.env")
if "NVIDIA_API_KEY" not in os.environ:
    raise ValueError("⚠️ NVIDIA_API_KEY not found in .env file!")

# Initialize Qdrant Client
qdrant_client = QdrantClient(host="localhost", port=6333)
TEXT_COLLECTION = "research_papers_text_v5"
IMAGE_COLLECTION = "research_papers_image_v6"

print("Loading models... This may take a minute.")

# Text Embedding Models
dense_text_model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1", device="cpu")
sparse_text_model = SparseTextEmbedding(model_name="Qdrant/bm25")

# Vision LLM (NVIDIA NIM)
vision_llm = ChatNVIDIA(model="nvidia/llama-3.1-nemotron-nano-vl-8b-v1")

# SigLIP Custom Wrapper (For Image Vectorstore Retrieval)
class SiglipEmbedder:
    def __init__(self, model_name="google/siglip-base-patch16-512", device="cpu"):
        self.processor = AutoProcessor.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(device)
        self.device = device
        
    def encode(self, text):
        if isinstance(text, str):
            text = [text]
        inputs = self.processor(text=text, padding="max_length", return_tensors="pt").to(self.device)
        with torch.no_grad():
            text_features = self.model.get_text_features(**inputs)
            text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)
        if len(text) == 1:
            return text_features[0].cpu().numpy()
        return text_features.cpu().numpy()

siglip_model = SiglipEmbedder(device="cpu")

# ==========================================
# 2. Retrieval Helper Functions
# ==========================================
def compute_weighted_fusion(list1, list2, alpha=0.4):
    scores_1 = {p.id: p.score for p in list1}
    scores_2 = {p.id: p.score for p in list2}
    
    def normalize(scores):
        if not scores: return scores
        min_val, max_val = min(scores.values()), max(scores.values())
        if max_val == min_val: return {k: 1.0 for k in scores}
        return {k: (v - min_val) / (max_val - min_val) for k, v in scores.items()}
        
    norm_1, norm_2 = normalize(scores_1), normalize(scores_2)
    fused_scores, point_map = {}, {}
    
    for p in list1:
        point_map[p.id] = p
        fused_scores[p.id] = alpha * norm_1.get(p.id, 0)
    for p in list2:
        point_map[p.id] = p
        fused_scores[p.id] = fused_scores.get(p.id, 0) + (1 - alpha) * norm_2.get(p.id, 0)
            
    sorted_ids = sorted(fused_scores.keys(), key=lambda x: fused_scores[x], reverse=True)
    return [point_map[pid] for pid in sorted_ids]

def retrieve_top_text_chunks(query, top_k=3):
    dense_vec = dense_text_model.encode(query)
    sparse_emb = list(sparse_text_model.embed([query]))[0]
    sparse_vec = models.SparseVector(
        indices=sparse_emb.indices.tolist(),
        values=sparse_emb.values.tolist()
    )

    dense_results = qdrant_client.query_points(
        collection_name=TEXT_COLLECTION,
        query=dense_vec.tolist(),
        using="dense", limit=10, with_payload=True
    ).points
    
    sparse_results = qdrant_client.query_points(
        collection_name=TEXT_COLLECTION,
        query=sparse_vec,
        using="sparse", limit=10, with_payload=True
    ).points
    
    # Apply alpha=0.4 fusion
    fused_results = compute_weighted_fusion(dense_results, sparse_results, alpha=0.4)
    
    # Return top K payloads (the text chunks)
    return [point.payload.get("text", "") for point in fused_results[:top_k]]

def retrieve_top_images(query, top_k=2):
    # Encode the text query into the SigLIP space
    query_vector = siglip_model.encode(query)
    
    results = qdrant_client.query_points(
        collection_name=IMAGE_COLLECTION,
        query=query_vector.tolist(),
        limit=top_k, 
        with_payload=True
    ).points
    
    # Extract the image paths from the payload
    image_paths = []
    for point in results:
        path = point.payload.get("image_path")
        if path and os.path.exists(path):
            image_paths.append(path)
    return image_paths

# ==========================================
# 3. Generation Function
# ==========================================
def encode_image_base64(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def generate_answer(query, text_chunks, image_paths):
    # 1. Format the retrieved context
    context_str = "\n\n---\n\n".join([f"Chunk {i+1}:\n{text}" for i, text in enumerate(text_chunks)])
    
    system_prompt = (
        "You are an expert AI research assistant. You have been provided with excerpts from a research paper "
        "and relevant figures/images extracted from the document.\n\n"
        "Use the provided Text Context and the provided Images to answer the user's question accurately.\n"
        f"TEXT CONTEXT:\n{context_str}"
    )

    # 2. Build the multimodal message content
    message_content = [
        {"type": "text", "text": f"{system_prompt}\n\nUSER QUESTION: {query}"}
    ]
    
    # Append the retrieved images to the prompt
    for img_path in image_paths:
        b64_img = encode_image_base64(img_path)
        img_ext = img_path.split('.')[-1].lower()
        mime_type = f"image/{img_ext}" if img_ext in ['jpeg', 'png'] else 'image/jpeg'
        
        message_content.append({
            "type": "image_url",
            "image_url": {"url": f"data:{mime_type};base64,{b64_img}"}
        })

    message = HumanMessage(content=message_content)
    
    # 3. Call the Vision LLM
    print("\nGenerating answer with Nemotron Nano VL...")
    response = vision_llm.invoke([message])
    return response.content

# ==========================================
# 4. Pipeline Execution
# ==========================================
def multimodal_rag_pipeline(query):
    print(f"\n[QUERY] {query}")
    
    # Step 1: Retrieve
    print("Retrieving top 3 text chunks...")
    top_texts = retrieve_top_text_chunks(query, top_k=3)
    
    print("Retrieving top 2 images...")
    top_images = retrieve_top_images(query, top_k=2)
    print(f"-> Found images: {top_images}")
    
    # Step 2: Generate
    answer = generate_answer(query, top_texts, top_images)
    
    print("\n===============================")
    print("🤖 LLM ANSWER:")
    print("===============================")
    print(answer)
    return answer

# --- RUN A TEST ---
if __name__ == "__main__":
    test_query = "What is the overall architecture proposed in this paper? Specifically, explain how the images map to the text."
    final_answer = multimodal_rag_pipeline(test_query)


I0000 00:00:1782920330.970848  136865 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782920331.608024  136865 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1782920333.915554  136865 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782920333.918426  136865 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

Loading models... This may take a minute.


KeyboardInterrupt: 